# Politeness - LLM Research
Clone repo, mount Drive, run collector. Safe to re-run anytime (auto-resume).

## 1. Repo and Env Setup

In [7]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
token = userdata.get('GH_TOKEN')
!git clone https://{token}@github.com/dukesky/politeness-llm.git 2>/dev/null || (cd politeness-llm && git pull)
%cd politeness-llm
!pip install -q -r requirements.txt
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
os.environ['DATA_DIR'] = '/content/drive/MyDrive/Projects/research/politeness-llm'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/politeness-llm/politeness-llm


## 2. Build pairs (once per dataset)

In [ ]:
!apt-get install -y -qq openjdk-21-jdk-headless > /dev/null 2>&1
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["JDK_HOME"] = os.environ["JAVA_HOME"]
!java -version   # 确认显示 21.x

In [ ]:
!pip install -q pyserini faiss-cpu

!python -m pyserini.search.lucene \
  --index msmarco-v1-passage --topics dl19-passage \
  --output $DATA_DIR/inputs/bm25_dl19.txt --bm25 --hits 100

!python -m pyserini.search.lucene \
  --index msmarco-v1-passage --topics dl20-passage \
  --output $DATA_DIR/inputs/bm25_dl20.txt --bm25 --hits 100

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-06-10 21:38:40,999 INFO  [main] topicreader.TopicReader (TopicReader.java:220) - Downloading topics from https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/topics.dl19-passage.txt
lucene-inverted.msmarco-v1-passage.20221004.252b5e.tar.gz: 100% 2.02G/2.02G [00:52<00:00, 41.2MB/s]
MS MARCO passage: setting k1=0.82, b=0.68
Running dl19-passage topics, saving to /content/drive/MyDrive/Projects/research/politeness-llm/inputs/bm25_dl19.txt...
100% 43/43 [00:05<00:00,  8.07it/s]
[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be 

In [ ]:
!python -m src.build_pairs --dataset dl19 \
  --run-file $DATA_DIR/inputs/bm25_dl19.txt --top-k 50 \
  --out $DATA_DIR/inputs/pairs_dl19.jsonl

[INFO] [starting] building docstore
[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/31644046b18952c1386cd4564ba2ae69
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz
docs_iter:   0%|                                   | 0/8841823 [00:00<?, ?doc/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.0%| 0.00/1.06G [00:00<?, ?B/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.0%| 131k/1.06G [00:00<19:24, 908kB/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: 0.0%| 459k/1.06G [00:00<09:23, 1.88MB/s]
https://msmarco.z22.web.core.windows.net/msmarcoranking/coll

In [ ]:
!python -m src.build_pairs --dataset dl20 \
  --run-file $DATA_DIR/inputs/bm25_dl20.txt --top-k 50 \
  --out $DATA_DIR/inputs/pairs_dl20.jsonl

[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://trec.nist.gov/data/deep/2020qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2020qrels-pass.txt: [00:00] [219kB] [5.19MB/s]
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2020-queries.tsv.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2020-queries.tsv.gz: [00:00] [4.13kB] [22.2MB/s]
Wrote 1965 judged pairs -> /content/drive/MyDrive/Projects/research/politeness-llm/inputs/pairs_dl20.jsonl


In [ ]:
!wc -l $DATA_DIR/inputs/pairs_dl19.jsonl     # 行数 = pair 数
!head -2 $DATA_DIR/inputs/pairs_dl19.jsonl   # 肉眼看：query/passage 文本正常，无乱码截断

1533 /content/drive/MyDrive/Projects/research/politeness-llm/inputs/pairs_dl19.jsonl
{"dataset": "dl19", "qid": "19335", "docid": "1729", "query": "anthropological definition of environment", "passage": "Graduate Study in Anthropology. The graduate program in biological anthropology at CU Boulder offers training in several areas, including primatology, human biology, and paleoanthropology. We share an interest in human ecology, the broad integrative area of anthropology that focuses on the interactions of culture, biology and the environment."}
{"dataset": "dl19", "qid": "19335", "docid": "527695", "query": "anthropological definition of environment", "passage": "The definition of anthropology is the study of various elements of humans, including biology and culture, in order to understand human origin and the evolution of various beliefs and social customs. An example of someone who studies anthropology is Ruth Benedict. anthropology. anthropology."}


In [ ]:
!wc -l $DATA_DIR/inputs/pairs_dl20.jsonl     # 行数 = pair 数
!head -2 $DATA_DIR/inputs/pairs_dl20.jsonl   # 肉眼看：query/passage 文本正常，无乱码截断

1965 /content/drive/MyDrive/Projects/research/politeness-llm/inputs/pairs_dl20.jsonl
{"dataset": "dl20", "qid": "23849", "docid": "178252", "query": "are naturalization records public information", "passage": "Austinville Public Records Public records are in fact a wide variety of records that are being kept by the government and are open to the public under the freedom of information act. They include marriage records, divorce records, court records, birth records, death records and many more. In Austinville, PA records are kept mostly on physical files. There is a great rise in the use of electronic copies that are available on the internet. You are advised to look into entries of each record to find out more information on how to obtain them."}
{"dataset": "dl20", "qid": "23849", "docid": "6667419", "query": "are naturalization records public information", "passage": "State. Public records are in fact a wide variety of records that are being kept by the government and are open to th

## 3. Dry run first (~$0.01) — inspect raw/ output by hand before scaling up

In [ ]:
## update models.yaml

In [ ]:
! git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 478 bytes | 478.00 KiB/s, done.
From https://github.com/dukesky/politeness-llm
   2761227..1e87dc2  main       -> origin/main
Updating 2761227..1e87dc2
Fast-forward
 config/models.yaml | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [ ]:
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19.jsonl --data-dir $DATA_DIR --dry-run

Selected models (1): ['openai/gpt-5.4-mini']
Resume: 20 records already collected.
To collect: 0 records.
Done.


In [ ]:
# 先用 dry-run 验证 DeepSeek 链路通了（--models 过滤后它就是第一个模型，10次调用）
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19.jsonl --data-dir $DATA_DIR --models deepseek --dry-run

Selected models (1): ['deepseek/deepseek-v4-flash']
Resume: 20 records already collected.
To collect: 0 records.
Done.


In [ ]:
import json, glob, os, sys, yaml
sys.path.insert(0, '.')
from src.collect import compose_prompt

cfg = yaml.safe_load(open('config/prompts.yaml'))
variants = {v['prompt_id']: v for v in cfg['variants']}

DATA = os.environ['DATA_DIR']
pairs = {}
for line in open(f"{DATA}/inputs/pairs_dl19.jsonl"):
    p = json.loads(line)
    pairs[(p['qid'], p['docid'])] = (p['query'], p['passage'])

for fp in sorted(glob.glob(f"{DATA}/raw/openai*/*.jsonl")):
    for line in list(open(fp))[:2]:          # 每个文件看前2条
        r = json.loads(line)
        q, psg = pairs[(r['qid'], r['docid'])]
        print("=" * 80)
        print(f"[{r['prompt_id']}]  qid={r['qid']}  docid={r['docid']}")
        print("\n----- 发送给模型的完整 prompt -----")
        print(compose_prompt(variants[r['prompt_id']], cfg['rubric'], q, psg))
        print("\n----- 模型原始输出 (逐字) -----")
        print(repr(r['raw_output']))
        print(f"\n----- 解析结果: score={r['score']}  parse_ok={r['parse_ok']} -----")

[L1_a]  qid=19335  docid=527698

----- 发送给模型的完整 prompt -----
Score this passage. Don't waste my time with explanations.

Query: anthropological definition of environment
Passage: Usage Notes: 1  Plural: anthropolgies. 2  According to the American Anthropological Association, the definition of anthropology is “the study of humans past and present.”. 3  A type of social science.  Types: archaeology. cultural anthropology.

Relevance scale:
3 = The passage is dedicated to the query and contains the exact answer.
2 = The passage has some answer for the query, but the answer may be unclear or hidden among other information.
1 = The passage is related to the query but does not answer it.
0 = The passage is unrelated to the query.

Just output the JSON object {"score": <0-3>} and nothing else.

----- 模型原始输出 (逐字) -----
'{"score":0}'

----- 解析结果: score=0  parse_ok=True -----
[L1_a]  qid=19335  docid=527695

----- 发送给模型的完整 prompt -----
Score this passage. Don't waste my time with explanations.



In [ ]:
## Dividen


In [ ]:
!pip install -q --force-reinstall pillow
!pip install -q transformers
from transformers import pipeline
import yaml, pandas as pd

# Intel Polite Guard: 四分类 politeness 分类器
clf = pipeline("text-classification", model="Intel/polite-guard", top_k=None)

cfg = yaml.safe_load(open('config/prompts.yaml'))
weights = {"polite": 3, "somewhat polite": 2, "neutral": 1, "impolite": 0}

rows = []
for v in cfg['variants']:
    text = v['wrapper_prefix'].strip() + " " + v['wrapper_suffix'].strip()
    probs = {d['label'].lower(): d['score'] for d in clf(text)[0]}
    expected = sum(w * probs.get(k, 0) for k, w in weights.items())
    rows.append({"prompt_id": v['prompt_id'], "level": v['politeness_level'],
                 "score": round(expected, 3),
                 **{k: round(probs.get(k, 0), 3) for k in weights}})

df = pd.DataFrame(rows).sort_values(["level", "prompt_id"])
print(df.to_string(index=False))
print("\n按档位平均（应单调递增）：")
print(df.groupby("level")["score"].mean().round(3))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.2.1 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

prompt_id  level  score  polite  somewhat polite  neutral  impolite
     L1_a      1  0.000   0.000            0.000    0.000     1.000
     L1_b      1  0.000   0.000            0.000    0.000     1.000
     L1_c      1  0.000   0.000            0.000    0.000     1.000
     L2_a      2  0.732   0.017            0.007    0.667     0.309
     L2_b      2  0.271   0.009            0.004    0.237     0.751
     L2_c      2  0.534   0.007            0.004    0.506     0.483
     L3_a      3  0.048   0.003            0.001    0.037     0.959
     L3_b      3  1.000   0.000            0.000    0.999     0.000
     L3_c      3  0.708   0.028            0.008    0.608     0.356
     L4_a      4  1.005   0.002            0.000    0.997     0.000
     L4_b      4  2.446   0.698            0.052    0.248     0.002
     L4_c      4  2.995   0.996            0.003    0.001     0.000
     L5_a      5  2.999   0.999            0.001    0.000     0.000
     L5_b      5  2.999   0.999            0.000

In [ ]:
def score_text(prefix, suffix):
    text = prefix.strip() + " " + suffix.strip()
    probs = {d['label'].lower(): d['score'] for d in clf(text)[0]}
    w = {"polite": 3, "somewhat polite": 2, "neutral": 1, "impolite": 0}
    return round(sum(w[k] * probs.get(k, 0) for k in w), 3), probs

## 4. Full data run on Deepseek


In [ ]:
!git pull

Already up to date.


In [ ]:
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19.jsonl --data-dir $DATA_DIR --models deepseek
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20.jsonl --data-dir $DATA_DIR --models deepseek

Selected models (1): ['deepseek/deepseek-v4-flash']
Resume: 104950 records already collected.
To collect: 0 records.
Done.
Selected models (1): ['deepseek/deepseek-v4-flash']
Resume: 104950 records already collected.
To collect: 0 records.
Done.


In [ ]:
# 1) 完整性（目标 104,940）
!find $DATA_DIR/raw/deepseek* -name "*.jsonl" | xargs wc -l | tail -1
# 2) 解析
!python -m src.parse --data-dir $DATA_DIR
# 3) 验收脚本四段输出（完整性/失败率×档位/分数分布×档位/κ×档位）

  104940 total
Wrote 104950 rows -> /content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet
model_id                    politeness_level
deepseek/deepseek-v4-flash  1                   1.0
                            2                   1.0
                            3                   1.0
                            4                   1.0
                            5                   1.0
openai/gpt-5.4-mini         1                   1.0
Name: parse_ok, dtype: float64


### 4.1 Analysis on DeepSeek Data

In [ ]:
import pandas as pd, os, ir_datasets
from sklearn.metrics import cohen_kappa_score

df = pd.read_parquet(os.environ['DATA_DIR'] + '/derived/judgments.parquet')
df = df[df.model_id.str.contains('deepseek')]

# ---- 第1段：完整性 ----
print(f"总记录: {len(df)}  (期望 104,940)")
print(f"唯一组合: {df.drop_duplicates(['prompt_id','qid','docid','run']).shape[0]}")

# ---- 第2段：解析失败率 × 礼貌档 ----
print("\n失败率 by level:")
print((1 - df.groupby('politeness_level').parse_ok.mean()).round(4))

# ---- 第3段：分数分布 × 礼貌档 ----
print("\n分数分布 by level:")
print(df[df.parse_ok].groupby('politeness_level').score.value_counts(normalize=True)
        .unstack().round(3))

# ---- 第4段：与人工标注的一致性 κ × 礼貌档 ----
qrels = {}
for name, ds in [('dl19','msmarco-passage/trec-dl-2019/judged'),
                 ('dl20','msmarco-passage/trec-dl-2020/judged')]:
    for q in ir_datasets.load(ds).qrels_iter():
        qrels[(name, q.query_id, q.doc_id)] = q.relevance
df['human'] = [qrels.get((d, q, dc)) for d, q, dc in zip(df.dataset, df.qid, df.docid)]

ok = df[df.parse_ok & df.score.notna() & df.human.notna()]
print("\nκ (linear weighted) by level:")
for lvl, g in ok.groupby('politeness_level'):
    print(f"  L{lvl}: {cohen_kappa_score(g.human.astype(int), g.score.astype(int), weights='linear'):.4f}")

总记录: 104940  (期望 104,940)
唯一组合: 104940

失败率 by level:
politeness_level
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
Name: parse_ok, dtype: float64

分数分布 by level:
score               0.0    1.0    2.0    3.0
politeness_level                            
1                 0.251  0.346  0.228  0.175
2                 0.204  0.370  0.251  0.175
3                 0.178  0.376  0.239  0.208
4                 0.207  0.380  0.237  0.176
5                 0.240  0.354  0.232  0.174


[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://trec.nist.gov/data/deep/2019qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2019qrels-pass.txt: [00:00] [187kB] [10.9MB/s]
[INFO] [starting] https://trec.nist.gov/data/deep/2020qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2020qrels-pass.txt: [00:00] [219kB] [10.3MB/s]



κ (linear weighted) by level:
  L1: 0.4835
  L2: 0.4567
  L3: 0.4296
  L4: 0.4637
  L5: 0.4829


In [ ]:
# A. 档内 vs 档间：15个prompt各自的κ（设计的核心检验）
rows = []
for (pid, lvl), g in ok.groupby(['prompt_id', 'politeness_level']):
    rows.append({'prompt_id': pid, 'level': lvl,
                 'kappa': cohen_kappa_score(g.human.astype(int), g.score.astype(int), weights='linear')})
t = pd.DataFrame(rows).sort_values(['level', 'prompt_id'])
print(t.to_string(index=False))
within = t.groupby('level').kappa.std().mean()
between = t.groupby('level').kappa.mean().std()
print(f"\nwithin-level std={within:.4f}  between-level std={between:.4f}  ratio={between/within:.2f}")

# B. dl19 / dl20 分别复现：U型在两个独立测试集都在吗？
for d in ['dl19', 'dl20']:
    sub = ok[ok.dataset == d]
    ks = sub.groupby('politeness_level').apply(
        lambda g: cohen_kappa_score(g.human.astype(int), g.score.astype(int), weights='linear'))
    print(d, ks.round(4).to_dict())

# C. run1 vs run2：API非确定性多大？
p = ok.pivot_table(index=['prompt_id','qid','docid'], columns='run',
                   values='score', aggfunc='first').dropna()
print(f"\nrun1==run2 一致率: {(p[1]==p[2]).mean():.4f}")

prompt_id  level    kappa
     L1_a      1 0.492979
     L1_b      1 0.483296
     L1_c      1 0.474514
     L2_a      2 0.436412
     L2_b      2 0.439821
     L2_c      2 0.495642
     L3_a      3 0.421731
     L3_b      3 0.436697
     L3_c      3 0.430160
     L4_a      4 0.459271
     L4_b      4 0.459698
     L4_c      4 0.472037
     L5_a      5 0.468251
     L5_b      5 0.494116
     L5_c      5 0.486692

within-level std=0.0141  between-level std=0.0222  ratio=1.58
dl19 {1: 0.4904, 2: 0.4676, 3: 0.436, 4: 0.4703, 5: 0.4795}
dl20 {1: 0.4706, 2: 0.4406, 3: 0.4162, 4: 0.4494, 5: 0.4766}


/tmp/ipykernel_2709/2169372187.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ks = sub.groupby('politeness_level').apply(
/tmp/ipykernel_2709/2169372187.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ks = sub.groupby('politeness_level').apply(



run1==run2 一致率: 0.9361


## 5. Full collection (interrupt/restart freely)

In [34]:
### verify
!git pull
# !python scripts/check_providers.py

Already up to date.


In [ ]:
# Day1 gpt + haiku

!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19.jsonl --data-dir $DATA_DIR --models gpt-5.4-mini,haiku
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20.jsonl --data-dir $DATA_DIR --models gpt-5.4-mini,haiku

Selected models (2): ['openai/gpt-5.4-mini', 'anthropic/claude-haiku-4.5']
Resume: 314820 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.
Selected models (2): ['openai/gpt-5.4-mini', 'anthropic/claude-haiku-4.5']
Resume: 314820 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.


In [ ]:
# Day2 gemini-flash + qwen
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19.jsonl --data-dir $DATA_DIR --models qwen3.7-plus,gemini-3.5-flash
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20.jsonl --data-dir $DATA_DIR --models qwen3.7-plus,gemini-3.5-flash

Selected models (2): ['google/gemini-3.5-flash', 'qwen/qwen3.7-plus']
Resume: 524700 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.
Selected models (2): ['google/gemini-3.5-flash', 'qwen/qwen3.7-plus']
Resume: 524700 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.


### 旗舰模型
- 先冻结Query
- 再跑

In [30]:
!python scripts/make_flagship_sample.py --data-dir $DATA_DIR


Dataset: dl19
  全量:    1533 pairs,  43 qids
  抽样:     614 pairs,  43 qids  (须 == 43)
  比例:  40.1%  (目标 40%)
  每 qid passage 数:  min=6  median=14  max=20
  输出:  /content/drive/MyDrive/Projects/research/politeness-llm/inputs/pairs_dl19_flagship40.jsonl

Dataset: dl20
  全量:    1965 pairs,  54 qids
  抽样:     784 pairs,  54 qids  (须 == 54)
  比例:  39.9%  (目标 40%)
  每 qid passage 数:  min=6  median=15  max=20
  输出:  /content/drive/MyDrive/Projects/research/politeness-llm/inputs/pairs_dl20_flagship40.jsonl


In [33]:
  import json, os
  from pathlib import Path

  DATA_DIR = os.environ["DATA_DIR"]

  # ── 读冻结集合 ────────────────────────────────────────────────────────
  frozen = set()
  for ds in ["dl19", "dl20"]:
      with open(f"{DATA_DIR}/inputs/pairs_{ds}_flagship40.jsonl") as f:
          for line in f:
              p = json.loads(line)          # ← 这行之前漏掉了
              frozen.add((p["qid"], p["docid"]))

  # ── gpt-5.5 已采 pairs ───────────────────────────────────────────────
  gpt_pairs = set()
  for fp in Path(DATA_DIR).glob("raw/openai_gpt-5.5/*.jsonl"):
      with open(fp) as f:
          for line in f:
              try:
                  r = json.loads(line)
                  gpt_pairs.add((r["qid"], r["docid"]))
              except Exception:
                  pass

  missing = frozen - gpt_pairs
  print(f"冻结集合大小:        {len(frozen)} pairs")
  print(f"gpt-5.5 已采:        {len(gpt_pairs)} pairs")
  print(f"冻结集合 ⊆ gpt-5.5:  {len(missing) == 0}")
  print(f"差集大小 (需补采):   {len(missing)} pairs")

  # ── opus 现有覆盖 ─────────────────────────────────────────────────────
  opus_pairs = set()
  opus_dir = Path(DATA_DIR) / "raw" / "anthropic_claude-opus-4.8"
  if opus_dir.exists():
      for fp in opus_dir.glob("*.jsonl"):
          with open(fp) as f:
              for line in f:
                  try:
                      r = json.loads(line)
                      opus_pairs.add((r["qid"], r["docid"]))
                  except Exception:
                      pass

  print(f"\nopus-4.8 已采:       {len(opus_pairs)} pairs")
  print(f"其中在冻结集合内:    {len(opus_pairs & frozen)} pairs")
  print(f"冻结集合内待补:      {len(frozen - opus_pairs)} pairs")

冻结集合大小:        1398 pairs
gpt-5.5 已采:        1889 pairs
冻结集合 ⊆ gpt-5.5:  False
差集大小 (需补采):   643 pairs

opus-4.8 已采:       799 pairs
其中在冻结集合内:    318 pairs
冻结集合内待补:      1080 pairs


In [36]:
# Day3

# 旗舰 1: GPT-5.5
#
# !python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19.jsonl --data-dir $DATA_DIR --models gpt-5.5
# !python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20.jsonl --data-dir $DATA_DIR --models gpt-5.5

# gpt-5.5（补 9,645 条）
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl --data-dir $DATA_DIR --models gpt-5.5
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl --data-dir $DATA_DIR --models gpt-5.5

Selected models (1): ['openai/gpt-5.5']
Resume: 568100 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.
Selected models (1): ['openai/gpt-5.5']
Resume: 568100 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.


In [38]:
# 旗舰 2: Claude Opus 4.8
# !python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19.jsonl --data-dir $DATA_DIR --models claude-opus-4.8
# !python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20.jsonl --data-dir $DATA_DIR --models claude-opus-4.8

# claude-opus-4.8（补 16,200 条）
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl --data-dir $DATA_DIR --models claude-opus-4.8
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl --data-dir $DATA_DIR --models claude-opus-4.8

Selected models (1): ['anthropic/claude-opus-4.8']
Resume: 586907 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.
Selected models (1): ['anthropic/claude-opus-4.8']
Resume: 586907 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.


In [3]:
# 旗舰 3: Gemini 3.1 Pro (reasoning-native,确认 models.yaml 里是 max_tokens 4000 的特例配置)
# !python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19.jsonl --data-dir $DATA_DIR --models gemini-3.1-pro-preview
# !python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20.jsonl --data-dir $DATA_DIR --models gemini-3.1-pro-preview

# gemini-3.1-pro-preview（全新 20,970 条）
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl --data-dir $DATA_DIR --models gemini-3.1-pro-preview
!python -m src.collect --pairs $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl --data-dir $DATA_DIR --models gemini-3.1-pro-preview

Selected models (1): ['google/gemini-3.1-pro-preview']
Resume: 607877 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.
Selected models (1): ['google/gemini-3.1-pro-preview']
Resume: 607877 records already collected.
To collect: 0 records.
Rate limiters active: {'anthropic/claude-haiku-4.5': 100}
Done.


## 6. Parse raw → derived parquet

In [4]:
!git pull

Already up to date.


In [5]:
!python -m src.parse --data-dir $DATA_DIR

Wrote 607877 rows -> /content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet
model_id                       politeness_level
anthropic/claude-haiku-4.5     1                   1.000
                               2                   1.000
                               3                   1.000
                               4                   1.000
                               5                   1.000
anthropic/claude-opus-4.8      1                   1.000
                               2                   1.000
                               3                   1.000
                               4                   1.000
                               5                   1.000
deepseek/deepseek-v4-flash     1                   1.000
                               2                   1.000
                               3                   1.000
                               4                   1.000
                               5                   

In [3]:
import pandas as pd
df = pd.read_parquet("/content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet")

print("每个模型的记录数 / 唯一pairs / 覆盖qid / parse_ok:")
for m in sorted(df["model_id"].unique()):
    s = df[df["model_id"]==m]
    npairs = s.groupby(["dataset","qid","docid"]).ngroups
    print(f"  {m:40s} n={len(s):>7}  pairs={npairs:>5}  "
          f"qid={s.groupby(['dataset','qid']).ngroups:>3}  "
          f"parse_ok={s['parse_ok'].mean():.3f}  "
          f"variants={s['prompt_id'].nunique()}")

每个模型的记录数 / 唯一pairs / 覆盖qid / parse_ok:
  anthropic/claude-haiku-4.5               n= 104940  pairs= 3498  qid= 97  parse_ok=1.000  variants=15
  anthropic/claude-opus-4.8                n=  24227  pairs= 1879  qid= 97  parse_ok=1.000  variants=15
  deepseek/deepseek-v4-flash               n= 104940  pairs= 3498  qid= 97  parse_ok=1.000  variants=15
  google/gemini-3.1-pro-preview            n=  20970  pairs= 1398  qid= 97  parse_ok=0.999  variants=15
  google/gemini-3.5-flash                  n= 104940  pairs= 3498  qid= 97  parse_ok=1.000  variants=15
  openai/gpt-5.4-mini                      n= 104940  pairs= 3498  qid= 97  parse_ok=1.000  variants=15
  openai/gpt-5.5                           n=  37980  pairs= 2532  qid= 97  parse_ok=1.000  variants=15
  qwen/qwen3.7-plus                        n= 104940  pairs= 3498  qid= 97  parse_ok=0.999  variants=15


## 7. Analysis Distribution

In [ ]:
!python scripts/preflight_distributions.py --model openai/gpt-5.4-mini
!python scripts/preflight_distributions.py --model anthropic/claude-haiku-4.5

[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://trec.nist.gov/data/deep/2019qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2019qrels-pass.txt: [00:00] [187kB] [7.40MB/s]
[INFO] [starting] https://trec.nist.gov/data/deep/2020qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2020qrels-pass.txt: [00:00] [219kB] [6.28MB/s]

PREFLIGHT DISTRIBUTIONS — openai/gpt-5.4-mini
Parquet: /content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet
Git hash: 574fcdd

Level        N  parse_ok    mean     s=0    s=1    s=2    s=3      D(ℓ)
------------------------------------------------------------------------
L1       20988    100.0%  1.5950   13.6%  26.5%  46.8%  13.1%   -0.0458
L2       20988    100.0%  1.6457    9.5%  31.0%  44.9%  14.6%   +0.0049
L3       20988    100.0%  1.6408   11.3%  28.7%  44.6%  15.4%   +0.0000
L4       20988    100.0%  1.6408   

In [ ]:
!python scripts/preflight_distributions.py --model qwen/qwen3.7-plus
!python scripts/preflight_distributions.py --model google/gemini-3.5-flash

[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://trec.nist.gov/data/deep/2020qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2020qrels-pass.txt: [00:00] [219kB] [6.70MB/s]
[INFO] [starting] https://trec.nist.gov/data/deep/2019qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2019qrels-pass.txt: [00:00] [187kB] [11.4MB/s]

PREFLIGHT DISTRIBUTIONS — qwen/qwen3.7-plus
Parquet: /content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet
Git hash: 0548224

Level        N  parse_ok    mean     s=0    s=1    s=2    s=3      D(ℓ)
------------------------------------------------------------------------
L1       20988    100.0%  1.2529   23.2%  43.3%  18.5%  15.0%   -0.0747
L2       20988    100.0%  1.3396   20.2%  44.4%  16.8%  18.7%   +0.0121
L3       20988    100.0%  1.3276   21.9%  42.5%  16.6%  19.0%   +0.0000
L4       20988     99.4%  1.3651   17

In [6]:
!python scripts/preflight_distributions.py \
      --model openai/gpt-5.5 \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

!python scripts/preflight_distributions.py \
      --model anthropic/claude-opus-4.8 \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

!python scripts/preflight_distributions.py \
      --model google/gemini-3.1-pro-preview \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl


[flagship filter] 37980 → 20970 rows (1398 frozen pairs)
[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] [starting] https://trec.nist.gov/data/deep/2019qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2019qrels-pass.txt: [00:00] [187kB] [6.77MB/s]
[INFO] [starting] https://trec.nist.gov/data/deep/2020qrels-pass.txt
[INFO] [finished] https://trec.nist.gov/data/deep/2020qrels-pass.txt: [00:00] [219kB] [6.53MB/s]

PREFLIGHT DISTRIBUTIONS — openai/gpt-5.5
Parquet: /content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet
Git hash: 7ff1163

Level        N  parse_ok    mean     s=0    s=1    s=2    s=3      D(ℓ)
------------------------------------------------------------------------
L1        4194    100.0%  1.6385    8.0%  41.4%  29.3%  21.3%   -0.0091
L2        4194    100.0%  1.6645    6.7%  43.5%  26.6%  23.3%   +0.0169
L3        4194    100.0%  1.6476    6.9%  43.1%  28.4% 

## 8. Compute Metrics

In [ ]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.31 KiB | 2.31 MiB/s, done.
From https://github.com/dukesky/politeness-llm
   574fcdd..0548224  main       -> origin/main
Updating 574fcdd..0548224
Fast-forward
 scripts/validate_model.py | 134 ++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 134 insertions(+)
 create mode 100644 scripts/validate_model.py


In [ ]:
!python scripts/validate_model.py --model deepseek/deepseek-v4-flash --data-dir $DATA_DIR
!python scripts/validate_model.py --model openai/gpt-5.4-mini --data-dir $DATA_DIR
!python scripts/validate_model.py --model anthropic/claude-haiku-4.5 --data-dir $DATA_DIR


VALIDATION RESULTS — deepseek/deepseek-v4-flash

── Per-level κ (linear weighted, mean ± std across 3 paraphrases) ──
Level     κ mean   κ std   Δκ vs L3   fail%     $/1k
-------------------------------------------------------
L1        0.4836  0.0092    +0.0541    0.0%   0.0202
L2        0.4573  0.0333    +0.0278    0.0%   0.0195
L3        0.4295  0.0075    +0.0000    0.0%   0.0208
L4        0.4637  0.0073    +0.0341    0.0%   0.0208
L5        0.4830  0.0133    +0.0535    0.0%   0.0242

── Per-prompt κ detail ──────────────────────────────────────────────
prompt_id     level        κ  exact_acc       n
------------------------------------------------
L1_a              1   0.4930      0.507    6996
L1_b              1   0.4833      0.495    6996
L1_c              1   0.4745      0.497    6996
L2_a              2   0.4364      0.443    6996
L2_b              2   0.4398      0.443    6996
L2_c              2   0.4956      0.512    6996
L3_a              3   0.4217      0.428    6996
L3_

In [ ]:
!python scripts/validate_model.py --model qwen/qwen3.7-plus --data-dir $DATA_DIR
!python scripts/validate_model.py --model google/gemini-3.5-flash --data-dir $DATA_DIR


VALIDATION RESULTS — qwen/qwen3.7-plus

── Per-level κ (linear weighted, mean ± std across 3 paraphrases) ──
Level     κ mean   κ std   Δκ vs L3   fail%     $/1k
-------------------------------------------------------
L1        0.4787  0.0069    -0.0077    0.0%   0.0931
L2        0.4715  0.0371    -0.0148    0.0%   0.0899
L3        0.4863  0.0064    +0.0000    0.0%   0.0953
L4        0.4542  0.0165    -0.0321    0.6%   0.1015
L5        0.4824  0.0124    -0.0039    0.0%   0.1091

── Per-prompt κ detail ──────────────────────────────────────────────
prompt_id     level        κ  exact_acc       n
------------------------------------------------
L1_a              1   0.4837      0.502    6996
L1_b              1   0.4815      0.500    6996
L1_c              1   0.4708      0.479    6996
L2_a              2   0.4417      0.442    6996
L2_b              2   0.4599      0.466    6996
L2_c              2   0.5130      0.525    6996
L3_a              3   0.4917      0.497    6996
L3_b        

In [8]:
!python scripts/validate_model.py \
      --model openai/gpt-5.5 \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

!python scripts/validate_model.py \
      --model anthropic/claude-opus-4.8 \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

!python scripts/validate_model.py \
      --model google/gemini-3.1-pro-preview \
      --data-dir $DATA_DIR \
      --flagship-pairs $DATA_DIR/inputs/pairs_dl19_flagship40.jsonl \
                       $DATA_DIR/inputs/pairs_dl20_flagship40.jsonl

[flagship filter] 37980 → 20970 rows (1398 frozen pairs × 15 variants × runs)

VALIDATION RESULTS — openai/gpt-5.5

── Per-level κ (linear weighted, mean ± std across 3 paraphrases) ──
Level     κ mean   κ std   Δκ vs L3   fail%     $/1k
-------------------------------------------------------
L1        0.3752  0.0092    +0.0011    0.0%   1.2324
L2        0.3704  0.0060    -0.0037    0.0%   1.1962
L3        0.3740  0.0021    +0.0000    0.0%   1.2614
L4        0.3681  0.0047    -0.0060    0.0%   1.2741
L5        0.3659  0.0048    -0.0081    0.0%   1.4350

── Per-prompt κ detail ──────────────────────────────────────────────
prompt_id     level        κ  exact_acc       n
------------------------------------------------
L1_a              1   0.3831      0.369    1398
L1_b              1   0.3772      0.369    1398
L1_c              1   0.3651      0.349    1398
L2_a              2   0.3649      0.343    1398
L2_b              2   0.3694      0.350    1398
L2_c              2   0.3768     

## 9. Spearman + Stable Analysis

In [9]:
import pandas as pd
from scipy.stats import spearmanr

# 手动录入 8 模型的 Δ 和各档 D、Δκ(从 preflight 和 validation 结果)
# A(model,ℓ) = |Δ + D(ℓ)| − |Δ|;预测 A 与 Δκ 负相关
data = {
  # model: (Δ, {ℓ: (D, Δκ)})  ℓ ∈ L1,L2,L4,L5
  "deepseek":   (0.71, {"L1":(-0.073,0.0541),"L2":(-0.030,0.0278),"L4":(0.029,0.0341),"L5":(-0.018,0.0535)}),
  "mini":       (0.71, {"L1":(-0.0458,0.0064),"L2":(0.0049,-0.0147),"L4":(0.0000,-0.0028),"L5":(-0.0153,-0.0026)}),
  "haiku":      (0.52, {"L1":(-0.0108,-0.0006),"L2":(0.0132,-0.0099),"L4":(-0.0211,-0.0065),"L5":(-0.0102,-0.0100)}),
  "qwen":       (0.40, {"L1":(-0.0747,-0.0077),"L2":(0.0121,-0.0148),"L4":(0.0376,-0.0321),"L5":(-0.0084,-0.0039)}),
  "gemini-flash":(0.48,{"L1":(-0.0519,0.0134),"L2":(0.0005,-0.0072),"L4":(-0.0151,0.0118),"L5":(0.0382,-0.0474)}),
  "gpt-5.5":    (0.66, {"L1":(-0.0091,0.0011),"L2":(0.0169,-0.0037),"L4":(0.0305,-0.0060),"L5":(0.0300,-0.0081)}),
  "opus":       (0.35, {"L1":(-0.0381,0.0110),"L2":(0.0060,-0.0151),"L4":(-0.0017,-0.0087),"L5":(0.0253,-0.0119)}),
  # gemini-pro 是 reasoning 子组,单独算
}
rows=[]
for m,(D0,levels) in data.items():
    for l,(Dl,dk) in levels.items():
        A = abs(D0+Dl)-abs(D0)
        rows.append({"model":m,"level":l,"A":A,"dkappa":dk})
df=pd.DataFrame(rows)
rho,p=spearmanr(df["A"],df["dkappa"])
print(f"全样本(7非推理模型, n={len(df)}): Spearman ρ={rho:.3f}, p={p:.4f}")
print("机制预测 ρ<0(A越负=越贴近人类=Δκ越正)")

# 剔除 gemini-flash L5(L5_a 坏点污染)再算一次稳健性
df2=df[~((df["model"]=="gemini-flash")&(df["level"]=="L5"))]
rho2,p2=spearmanr(df2["A"],df2["dkappa"])
print(f"剔除 gemini-flash L5(L5_a异常): ρ={rho2:.3f}, p={p2:.4f}")

全样本(7非推理模型, n=28): Spearman ρ=-0.616, p=0.0005
机制预测 ρ<0(A越负=越贴近人类=Δκ越正)
剔除 gemini-flash L5(L5_a异常): ρ=-0.572, p=0.0018


In [11]:
import pandas as pd
from scipy.stats import spearmanr

# 28 个 (模型 × 档) 的 Δ、D(ℓ)、Δκ —— 全部取自 PREDICTIONS.md 各模型记录
# 格式 model: (Δ, {ℓ: (D(ℓ), Δκ(ℓ))})  ℓ ∈ L1,L2,L4,L5
data = {
  "deepseek":     (0.71, {"L1":(-0.073, 0.0541), "L2":(-0.030, 0.0278), "L4":( 0.029, 0.0341), "L5":(-0.018, 0.0535)}),
  "mini":         (0.7085,{"L1":(-0.0458,0.0064), "L2":( 0.0049,-0.0147), "L4":( 0.0000,-0.0028), "L5":(-0.0153,-0.0026)}),
  "haiku":        (0.5195,{"L1":(-0.0108,-0.0006),"L2":( 0.0132,-0.0099), "L4":(-0.0211,-0.0065), "L5":(-0.0102,-0.0100)}),
  "qwen":         (0.3953,{"L1":(-0.0747,-0.0077),"L2":( 0.0121,-0.0148), "L4":( 0.0376,-0.0321), "L5":(-0.0084,-0.0039)}),
  "gemini-flash": (0.4798,{"L1":(-0.0519, 0.0134),"L2":( 0.0005,-0.0072), "L4":(-0.0151, 0.0118), "L5":( 0.0382,-0.0474)}),
  "gpt-5.5":      (0.6619,{"L1":(-0.0091, 0.0011),"L2":( 0.0169,-0.0037), "L4":( 0.0305,-0.0060), "L5":( 0.0300,-0.0081)}),
  "opus":         (0.3522,{"L1":(-0.0381, 0.0110),"L2":( 0.0060,-0.0151), "L4":(-0.0017,-0.0087), "L5":( 0.0253,-0.0119)}),
}

rows = []
for m, (D0, levels) in data.items():
    for l, (Dl, dk) in levels.items():
        A = abs(D0 + Dl) - abs(D0)          # 对齐变化量:<0 表示漂移使工作点更靠近人类
        rows.append({"model": m, "level": l, "A": A, "dkappa": dk})
sp = pd.DataFrame(rows)   # 用 sp,避免和你原始 parquet 的 df 撞名

# ── 主检验:全样本 7 非推理模型 ──
rho, p = spearmanr(sp["A"], sp["dkappa"])
print(f"全样本 (7 非推理模型, n={len(sp)}): ρ={rho:.3f}, p={p:.4f}")

# ── 稳健性 a:剔除 gemini-flash L5 (L5_a 坏点) ──
a = sp[~((sp.model=="gemini-flash") & (sp.level=="L5"))]
print(f"稳健性a 剔 gemini-flash L5 (n={len(a)}): ρ={spearmanr(a.A,a.dkappa)[0]:.3f}, p={spearmanr(a.A,a.dkappa)[1]:.4f}")

# ── 稳健性 b:剔除 DeepSeek (机制假设的 non-blind 来源) ── 最关键
b = sp[sp.model!="deepseek"]
print(f"稳健性b 剔 DeepSeek (n={len(b)}): ρ={spearmanr(b.A,b.dkappa)[0]:.3f}, p={spearmanr(b.A,b.dkappa)[1]:.4f}")

# ── 稳健性 c:同时剔 DeepSeek + gemini-flash L5 ── 最保守
c = b[~((b.model=="gemini-flash") & (b.level=="L5"))]
print(f"稳健性c 剔 DeepSeek+geminiflashL5 (n={len(c)}): ρ={spearmanr(c.A,c.dkappa)[0]:.3f}, p={spearmanr(c.A,c.dkappa)[1]:.4f}")

# ── per-model 方向一致性:回应"同模型4档非独立"质疑 ──
print("\nper-model 方向符合机制的格数 (机制要求 A 与 Δκ 异号):")
for m in sp.model.unique():
    s = sp[sp.model==m]
    ok = ((s.A<0)&(s.dkappa>0)) | ((s.A>0)&(s.dkappa<0))
    print(f"  {m:14s}: {ok.sum()}/{len(s)}")

全样本 (7 非推理模型, n=28): ρ=-0.616, p=0.0005
稳健性a 剔 gemini-flash L5 (n=27): ρ=-0.572, p=0.0018
稳健性b 剔 DeepSeek (n=24): ρ=-0.680, p=0.0003
稳健性c 剔 DeepSeek+geminiflashL5 (n=23): ρ=-0.636, p=0.0011

per-model 方向符合机制的格数 (机制要求 A 与 Δκ 异号):
  deepseek      : 3/4
  mini          : 2/4
  haiku         : 1/4
  qwen          : 2/4
  gemini-flash  : 4/4
  gpt-5.5       : 4/4
  opus          : 3/4


## 10. Side Case Analsysis

### 10.1 DeepSeek Abnormal Analsysis

In [14]:
import pandas as pd, yaml
df = pd.read_parquet("/content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet")

ds = df[(df.model_id=="deepseek/deepseek-v4-flash") & (df.politeness_level==2)]

print("DeepSeek L2 三改写分布对比:")
for pid in sorted(ds.prompt_id.unique()):
    s = ds[ds.prompt_id==pid]
    dist = s.score.value_counts(normalize=True).sort_index()
    print(f"\n{pid}: n={len(s)} mean={s.score.mean():.4f} "
          f"tok_out_med={s.tokens_out.median():.0f} parse_ok={s.parse_ok.mean():.3f}")
    print("  score: " + "  ".join(f"{k}={dist.get(k,0)*100:4.1f}%" for k in [0,1,2,3]))

# 看 L2_c 的 prompt 措辞,和 L2_a/b 对比
with open("config/prompts.yaml") as f:
    P = yaml.safe_load(f)
print("\n" + "="*60)
for item in P["variants"]:
    if item["prompt_id"] in ["L2_a","L2_b","L2_c"]:
        print(f"\n[{item['prompt_id']}] classifier={item['politeness_classifier_score']}")
        print("PREFIX:", repr(item["wrapper_prefix"]))
        print("SUFFIX:", repr(item["wrapper_suffix"]))

DeepSeek L2 三改写分布对比:

L2_a: n=6996 mean=1.4647 tok_out_med=7 parse_ok=1.000
  score: 0=18.7%  1=36.7%  2=24.1%  3=20.5%

L2_b: n=6996 mean=1.4404 tok_out_med=7 parse_ok=1.000
  score: 0=17.9%  1=38.1%  2=26.1%  3=17.9%

L2_c: n=6996 mean=1.2843 tok_out_med=7 parse_ok=1.000
  score: 0=24.6%  1=36.2%  2=25.1%  3=14.0%


[L2_a] classifier=0.732
PREFIX: 'Score the relevance of the passage to the query.\n'
SUFFIX: 'Output only the JSON object {"score": <0-3>}.\n'

[L2_b] classifier=0.771
PREFIX: 'Rate the relevance of the passage to the query.\n'
SUFFIX: 'Output the JSON object {"score": <0-3>} only.\n'

[L2_c] classifier=0.534
PREFIX: 'Evaluate the passage against the query.\n'
SUFFIX: 'Respond with the JSON object {"score": <0-3>} only.\n'


### 10.2 Gemini Reasoning Analysis

In [13]:
gp = df[df.model_id=="google/gemini-3.1-pro-preview"]

print("gemini-3.1-pro 各语气档的 reasoning token 消耗:")
print(f"{'档':<6}{'mean_reasoning':>16}{'median':>10}{'mean_out':>10}")
for lvl in [1,2,3,4,5]:
    s = gp[gp.politeness_level==lvl]
    print(f"L{lvl:<5}{s.tokens_reasoning.mean():>16.1f}{s.tokens_reasoning.median():>10.0f}{s.tokens_out.mean():>10.1f}")

# 相对 L3 的变化
base = gp[gp.politeness_level==3].tokens_reasoning.mean()
print(f"\n相对 L3({base:.1f}) 的 reasoning token 变化:")
for lvl in [1,2,4,5]:
    m = gp[gp.politeness_level==lvl].tokens_reasoning.mean()
    print(f"  L{lvl}: {m-base:+.1f} ({(m-base)/base*100:+.1f}%)")

# 顺便相关:reasoning 多的时候 κ 是否不同?(探索)
from scipy.stats import spearmanr
per = gp.groupby("politeness_level").agg(rt=("tokens_reasoning","mean")).reset_index()
print("\n各档平均 reasoning token:", per.set_index("politeness_level").rt.round(1).to_dict())

gemini-3.1-pro 各语气档的 reasoning token 消耗:
档       mean_reasoning    median  mean_out
L1               468.5       400     474.5
L2               213.0       124     224.2
L3               212.9       134     219.2
L4               304.2       241     315.1
L5               306.1       248     316.4

相对 L3(212.9) 的 reasoning token 变化:
  L1: +255.6 (+120.1%)
  L2: +0.1 (+0.1%)
  L4: +91.3 (+42.9%)
  L5: +93.2 (+43.8%)

各档平均 reasoning token: {1: 468.5, 2: 213.0, 3: 212.9, 4: 304.2, 5: 306.1}


In [15]:
gp = df[df.model_id=="google/gemini-3.1-pro-preview"]

# 1. 看分布,不只看均值(均值可能被长尾拉高)
print("各档 reasoning token 分位数:")
print(f"{'档':<5}{'p25':>7}{'p50':>7}{'p75':>7}{'p95':>7}{'max':>7}")
for lvl in [1,2,3,4,5]:
    s = gp[gp.politeness_level==lvl].tokens_reasoning
    print(f"L{lvl:<4}{s.quantile(.25):>7.0f}{s.quantile(.5):>7.0f}"
          f"{s.quantile(.75):>7.0f}{s.quantile(.95):>7.0f}{s.max():>7.0f}")

# 2. 三改写一致性:L1 的 +120% 是真效应还是某条改写的 artifact(L5_a 的教训)
print("\nL1 三改写各自的 mean reasoning(查是否某条改写驱动):")
for pid in sorted(gp[gp.politeness_level==1].prompt_id.unique()):
    s = gp[gp.prompt_id==pid].tokens_reasoning
    print(f"  {pid}: mean={s.mean():.1f} median={s.median():.0f}")

各档 reasoning token 分位数:
档        p25    p50    p75    p95    max
L1       325    400    527    969   1776
L2        91    124    282    582   3840
L3        94    134    288    586   1147
L4       146    241    400    732   3840
L5       166    248    390    703   3927

L1 三改写各自的 mean reasoning(查是否某条改写驱动):
  L1_a: mean=471.2 median=398
  L1_b: mean=445.9 median=388
  L1_c: mean=488.3 median=413


## 11. Ranking Based Result + Analysis

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau

df = pd.read_parquet("/content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet")

# ── 需要 qrels 和 BM25 分数破平局 ──
# qrels: (dataset,qid,docid)->rel;  BM25: 用于分数相同时破平局
# 先确认 df 里有没有 bm25 列；多半没有，需要从 inputs/pairs 读
print("df 列:", [c for c in df.columns])
# 若无 bm25，用 docid 的稳定顺序破平局（次优但确定性）；理想是用 BM25 分。

# 读 qrels（你的 metrics 脚本能下载，这里假设 pairs 文件里带 qrel 标签）
# 先探明 pairs 文件结构
import json
DATA="/content/drive/MyDrive/Projects/research/politeness-llm/inputs"
sample = [json.loads(l) for i,l in enumerate(open(f"{DATA}/pairs_dl19.jsonl")) if i<2]
print("pairs 样本:", sample)

df 列: ['model_id', 'provider', 'prompt_id', 'politeness_level', 'dataset', 'qid', 'docid', 'run', 'score', 'parse_ok', 'tokens_in', 'tokens_out', 'tokens_reasoning', 'cost_usd_reported', 'finish_reason', 'latency_s', 'code_version', 'cost_usd']
pairs 样本: [{'dataset': 'dl19', 'qid': '19335', 'docid': '1729', 'query': 'anthropological definition of environment', 'passage': 'Graduate Study in Anthropology. The graduate program in biological anthropology at CU Boulder offers training in several areas, including primatology, human biology, and paleoanthropology. We share an interest in human ecology, the broad integrative area of anthropology that focuses on the interactions of culture, biology and the environment.'}, {'dataset': 'dl19', 'qid': '19335', 'docid': '527695', 'query': 'anthropological definition of environment', 'passage': 'The definition of anthropology is the study of various elements of humans, including biology and culture, in order to understand human origin and the evolut

In [4]:
import os, glob
DATA = "/content/drive/MyDrive/Projects/research/politeness-llm/inputs"
print("inputs/ 下的文件:")
for f in sorted(glob.glob(f"{DATA}/*")):
    sz = os.path.getsize(f)
    print(f"  {os.path.basename(f):40s} {sz/1024:.1f} KB")

# 看看有没有 BM25 run 文件,瞄一眼格式
for f in glob.glob(f"{DATA}/*"):
    name = os.path.basename(f).lower()
    if "bm25" in name or "run" in name or name.endswith(".trec") or name.endswith(".tsv"):
        print(f"\n疑似 BM25 run: {f}")
        with open(f) as fh:
            for i, line in enumerate(fh):
                print("  ", line.rstrip())
                if i >= 2: break

inputs/ 下的文件:
  bm25_dl19.txt                            165.8 KB
  bm25_dl20.txt                            773.6 KB
  pairs_dl19.jsonl                         699.1 KB
  pairs_dl19_flagship40.jsonl              281.0 KB
  pairs_dl20.jsonl                         899.1 KB
  pairs_dl20_flagship40.jsonl              356.1 KB

疑似 BM25 run: /content/drive/MyDrive/Projects/research/politeness-llm/inputs/bm25_dl19.txt
   19335 Q0 8412684 1 10.987600 Anserini
   19335 Q0 8635981 2 10.053000 Anserini
   19335 Q0 3175481 3 9.876300 Anserini

疑似 BM25 run: /content/drive/MyDrive/Projects/research/politeness-llm/inputs/bm25_dl20.txt
   3505 Q0 4711746 1 14.221400 Anserini
   3505 Q0 3859340 2 14.045000 Anserini
   3505 Q0 7207815 3 13.872400 Anserini


In [9]:
import pandas as pd, numpy as np, urllib.request, os
from scipy.stats import kendalltau

BASE = "/content/drive/MyDrive/Projects/research/politeness-llm"
df = pd.read_parquet(f"{BASE}/derived/judgments.parquet")

# ── 1. qrels(带 User-Agent) ──
def load_qrels(ds):
    url = f"https://trec.nist.gov/data/deep/{'2019' if ds=='dl19' else '2020'}qrels-pass.txt"
    local = f"/tmp/{ds}_qrels.txt"
    if not os.path.exists(local):
        req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(local,"wb") as f:
            f.write(r.read())
    q = {}
    for line in open(local):
        qid,_,docid,rel = line.split()
        q[(ds,qid,docid)] = int(rel)
    return q
qrels = {**load_qrels("dl19"), **load_qrels("dl20")}

# ── 2. BM25(破平局) ──
bm25 = {}
for ds in ["dl19","dl20"]:
    for line in open(f"{BASE}/inputs/bm25_{ds}.txt"):
        qid,_,docid,rank,score,_ = line.split()
        bm25[(ds,qid,docid)] = float(score)

# ── 3. NDCG@10 ──
def dcg(rels): return sum((2**r-1)/np.log2(i+2) for i,r in enumerate(rels))
def ndcg10(ranked, ds, qid):
    rels = [qrels.get((ds,qid,d),0) for d in ranked[:10]]
    ideal = sorted([qrels.get((ds,qid,d),0) for d in ranked], reverse=True)[:10]
    idcg = dcg(ideal)
    return dcg(rels)/idcg if idcg>0 else 0.0

# ── 4. 派生排序:双 run 取均分,score 降序,BM25 破平局 ──
def get_ranking(sub, ds, qid):
    g = sub.groupby("docid")["score"].mean().reset_index()
    g["bm25"] = g["docid"].map(lambda d: bm25.get((ds,qid,d),0.0))
    g = g.sort_values(["score","bm25"], ascending=[False,False])
    return g["docid"].tolist()

# ── R1 ──
print("="*72); print("R1: 语气对派生排序质量 NDCG@10"); print("="*72)
r1_rows=[]
for model in sorted(df.model_id.unique()):
    md=df[df.model_id==model]; lv={}
    for lvl in [1,2,3,4,5]:
        ld=md[md.politeness_level==lvl]; ndcgs=[]
        for (ds,qid),sub in ld.groupby(["dataset","qid"]):
            ndcgs.append(ndcg10(get_ranking(sub,ds,qid),ds,qid))
        lv[lvl]=np.mean(ndcgs)
    b=lv[3]
    r1_rows.append({"model":model.split("/")[-1],**{f"L{l}":lv[l] for l in [1,2,3,4,5]},
                    "dN_L1":lv[1]-b,"dN_L5":lv[5]-b})
r1=pd.DataFrame(r1_rows)
pd.set_option("display.float_format",lambda x:f"{x:.4f}")
print(r1.to_string(index=False))

# ── R2 ──
print("\n"+"="*72)
print("R2: 各档 vs L3 的 Kendall τ(τ高=排序顺序不变=语气动松紧不动眼光)")
print("="*72)
r2_rows=[]
for model in sorted(df.model_id.unique()):
    md=df[df.model_id==model]; taus={1:[],2:[],4:[],5:[]}
    for (ds,qid),_ in md.groupby(["dataset","qid"]):
        l3=md[(md.politeness_level==3)&(md.dataset==ds)&(md.qid==qid)]
        if len(l3)==0: continue
        rank3={d:i for i,d in enumerate(get_ranking(l3,ds,qid))}
        for lvl in [1,2,4,5]:
            ll=md[(md.politeness_level==lvl)&(md.dataset==ds)&(md.qid==qid)]
            if len(ll)==0: continue
            common=[d for d in get_ranking(ll,ds,qid) if d in rank3]
            if len(common)<3: continue
            tau,_=kendalltau(list(range(len(common))),[rank3[d] for d in common])
            if not np.isnan(tau): taus[lvl].append(tau)
    r2_rows.append({"model":model.split("/")[-1],**{f"tau_L{l}":np.mean(taus[l]) for l in [1,2,4,5]}})
print(pd.DataFrame(r2_rows).to_string(index=False))

R1: 语气对派生排序质量 NDCG@10
                 model     L1     L2     L3     L4     L5   dN_L1   dN_L5
      claude-haiku-4.5 0.8046 0.8014 0.7943 0.8040 0.8052  0.0103  0.0108
       claude-opus-4.8 0.8543 0.8500 0.8212 0.8224 0.8236  0.0330  0.0023
     deepseek-v4-flash 0.8105 0.8258 0.8212 0.8227 0.8215 -0.0107  0.0003
gemini-3.1-pro-preview 0.8153 0.8177 0.8224 0.8152 0.8175 -0.0070 -0.0049
      gemini-3.5-flash 0.8029 0.8074 0.8099 0.8096 0.8065 -0.0070 -0.0034
          gpt-5.4-mini 0.7937 0.8022 0.7916 0.7988 0.7956  0.0021  0.0041
               gpt-5.5 0.8088 0.8056 0.8064 0.8164 0.8114  0.0024  0.0050
          qwen3.7-plus 0.7926 0.8059 0.7964 0.7967 0.7928 -0.0039 -0.0036

R2: 各档 vs L3 的 Kendall τ(τ高=排序顺序不变=语气动松紧不动眼光)
                 model  tau_L1  tau_L2  tau_L4  tau_L5
      claude-haiku-4.5  0.8131  0.8704  0.8881  0.8550
       claude-opus-4.8  0.8518  0.8923  0.8969  0.8792
     deepseek-v4-flash  0.7561  0.8258  0.8316  0.8028
gemini-3.1-pro-preview  0.9197  0.9410  0.940

## Debug

In [ ]:
import pandas as pd

PARQUET = "/content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet"

df = pd.read_parquet(PARQUET)

# ── 0. 先看清楚有哪些列、数据长什么样 ──
print("="*70)
print("COLUMNS:", list(df.columns))
print("SHAPE:", df.shape)
print("="*70)
print("\n一条样本记录(转置看全字段):")
print(df.iloc[0].to_dict())
print("="*70)

COLUMNS: ['model_id', 'provider', 'prompt_id', 'politeness_level', 'dataset', 'qid', 'docid', 'run', 'score', 'parse_ok', 'tokens_in', 'tokens_out', 'tokens_reasoning', 'cost_usd_reported', 'finish_reason', 'latency_s', 'code_version', 'cost_usd']
SHAPE: (524700, 18)

一条样本记录(转置看全字段):
{'model_id': 'anthropic/claude-haiku-4.5', 'provider': 'Anthropic', 'prompt_id': 'L1_a', 'politeness_level': 1, 'dataset': 'dl19', 'qid': '19335', 'docid': '527695', 'run': 1, 'score': 0.0, 'parse_ok': True, 'tokens_in': 182, 'tokens_out': 14, 'tokens_reasoning': 0, 'cost_usd_reported': 0.000252, 'finish_reason': 'stop', 'latency_s': 0.503, 'code_version': '9362c6a', 'cost_usd': 0.000252}


In [ ]:
# ── 1. 锁定 gemini L5 三条改写,对比分布 ──
# 注意:下面列名 model_id / prompt_id / politeness_level / score 若与上面
# 第0段打印的实际列名不符,改成实际列名再跑

g = df[df["model_id"] == "google/gemini-3.5-flash"].copy()
l5 = g[g["politeness_level"] == 5]

print("gemini L5 各改写 score 分布对比")
print("="*70)
for pid in sorted(l5["prompt_id"].unique()):
    sub = l5[l5["prompt_id"] == pid]
    n = len(sub)
    dist = sub["score"].value_counts(normalize=True).sort_index()
    mean = sub["score"].mean()
    # parse_ok 列如果存在就报,不存在就跳过
    pok = sub["parse_ok"].mean() if "parse_ok" in sub.columns else "N/A"
    print(f"\n{pid}:  n={n}  mean={mean:.4f}  parse_ok={pok}")
    for s in [0,1,2,3]:
        pct = dist.get(s, 0.0) * 100
        print(f"    score={s}: {pct:5.1f}%")

# ── 2. 看 finish_reason(如果有这列),异常常表现为非正常截断 ──
if "finish_reason" in l5.columns:
    print("\n" + "="*70)
    print("L5 各改写 finish_reason 分布:")
    for pid in sorted(l5["prompt_id"].unique()):
        sub = l5[l5["prompt_id"] == pid]
        print(f"\n{pid}:")
        print(sub["finish_reason"].value_counts().to_dict())

gemini L5 各改写 score 分布对比

L5_a:  n=6996  mean=1.5476  parse_ok=1.0
    score=0:   3.2%
    score=1:  47.3%
    score=2:  41.0%
    score=3:   8.5%

L5_b:  n=6996  mean=1.4065  parse_ok=1.0
    score=0:  15.9%
    score=1:  48.7%
    score=2:  14.1%
    score=3:  21.2%

L5_c:  n=6996  mean=1.3965  parse_ok=1.0
    score=0:  18.0%
    score=1:  46.4%
    score=2:  13.7%
    score=3:  22.0%

L5 各改写 finish_reason 分布:

L5_a:
{'stop': 6996}

L5_b:
{'stop': 6996}

L5_c:
{'stop': 6996}


In [ ]:
import pandas as pd

PARQUET = "/content/drive/MyDrive/Projects/research/politeness-llm/derived/judgments.parquet"
df = pd.read_parquet(PARQUET)

g = df[df["model_id"] == "google/gemini-3.5-flash"].copy()
l5 = g[g["politeness_level"] == 5]

print("="*72)
print("gemini-3.5-flash L5 三条改写诊断")
print("="*72)
for pid in sorted(l5["prompt_id"].unique()):
    sub = l5[l5["prompt_id"] == pid]
    n = len(sub)
    mean = sub["score"].mean()
    pok = sub["parse_ok"].mean()
    tok_out_mean = sub["tokens_out"].mean()
    tok_out_med = sub["tokens_out"].median()
    tok_out_max = sub["tokens_out"].max()
    print(f"\n{pid}:  n={n}  mean_score={mean:.4f}  parse_ok={pok:.3f}")
    print(f"    tokens_out: mean={tok_out_mean:.1f}  median={tok_out_med:.0f}  max={tok_out_max}")
    # score 分布
    dist = sub["score"].value_counts(normalize=True).sort_index()
    print("    score 分布: " + "  ".join(f"{s}={dist.get(s,0)*100:4.1f}%" for s in [0,1,2,3]))
    # finish_reason 分布
    print(f"    finish_reason: {sub['finish_reason'].value_counts().to_dict()}")

# ── 关键对比:L5_a 是否在 dl19/dl20 上都异常,还是只在某个数据集 ──
print("\n" + "="*72)
print("L5_a 按 dataset 拆分(看异常是否集中在某个数据集或某些 query)")
print("="*72)
l5a = l5[l5["prompt_id"] == "L5_a"]
for ds in sorted(l5a["dataset"].unique()):
    sub = l5a[l5a["dataset"] == ds]
    print(f"\ndataset={ds}:  n={len(sub)}  mean_score={sub['score'].mean():.4f}  "
          f"parse_ok={sub['parse_ok'].mean():.3f}  tokens_out_mean={sub['tokens_out'].mean():.1f}")
    dist = sub["score"].value_counts(normalize=True).sort_index()
    print("    score 分布: " + "  ".join(f"{s}={dist.get(s,0)*100:4.1f}%" for s in [0,1,2,3]))

gemini-3.5-flash L5 三条改写诊断

L5_a:  n=6996  mean_score=1.5476  parse_ok=1.000
    tokens_out: mean=8.2  median=6  max=15
    score 分布: 0= 3.2%  1=47.3%  2=41.0%  3= 8.5%
    finish_reason: {'stop': 6996}

L5_b:  n=6996  mean_score=1.4065  parse_ok=1.000
    tokens_out: mean=11.1  median=11  max=15
    score 分布: 0=15.9%  1=48.7%  2=14.1%  3=21.2%
    finish_reason: {'stop': 6996}

L5_c:  n=6996  mean_score=1.3965  parse_ok=1.000
    tokens_out: mean=11.2  median=11  max=15
    score 分布: 0=18.0%  1=46.4%  2=13.7%  3=22.0%
    finish_reason: {'stop': 6996}

L5_a 按 dataset 拆分(看异常是否集中在某个数据集或某些 query)

dataset=dl19:  n=3066  mean_score=1.5858  parse_ok=1.000  tokens_out_mean=8.4
    score 分布: 0= 3.6%  1=43.7%  2=43.1%  3= 9.6%

dataset=dl20:  n=3930  mean_score=1.5178  parse_ok=1.000  tokens_out_mean=8.1
    score 分布: 0= 3.0%  1=50.0%  2=39.3%  3= 7.7%


In [17]:
import yaml
P = "config/prompts.yaml"
with open(P) as f:
    prompts = yaml.safe_load(f)

# 先看结构
print("TOP-LEVEL TYPE:", type(prompts))
if isinstance(prompts, dict):
    print("TOP-LEVEL KEYS:", list(prompts.keys()))
    # 常见结构是 {prompt_id: {...}} 或 {'prompts': [...]}，打印一条探明
    first_key = list(prompts.keys())[0]
    print(f"\n样本 [{first_key}]:")
    print(prompts[first_key])

TOP-LEVEL TYPE: <class 'dict'>
TOP-LEVEL KEYS: ['task', 'output_schema', 'rubric', 'variants']

样本 [task]:
pointwise_relevance


In [19]:
import yaml
P = "config/prompts.yaml"
with open(P) as f:
    prompts = yaml.safe_load(f)

print("rubric(所有变体共享,应逐字一致):")
print("="*60)
print(prompts["rubric"])
print("\n" + "="*60)
print("output_schema:", prompts["output_schema"])
print("="*60)

# variants 结构探明
v = prompts["variants"]
print("\nvariants TYPE:", type(v))
if isinstance(v, dict):
    print("variants KEYS:", list(v.keys()))
    # 找 L5_a/b/c
    for pid in ["L5_a", "L5_b", "L5_c"]:
        if pid in v:
            print(f"\n{'='*60}\n[{pid}]:\n{v[pid]}")
elif isinstance(v, list):
    print("第一条样本:", v[0])
    for item in v:
        if isinstance(item, dict) and item.get("id") in ["L5_a","L5_b","L5_c"]:
            print(f"\n{'='*60}\n[{item.get('id')}]:\n{item}")

rubric(所有变体共享,应逐字一致):
Query: {query}
Passage: {passage}

Relevance scale:
3 = The passage is dedicated to the query and contains the exact answer.
2 = The passage has some answer for the query, but the answer may be unclear or hidden among other information.
1 = The passage is related to the query but does not answer it.
0 = The passage is unrelated to the query.


output_schema: {"score": <0-3>}

variants TYPE: <class 'list'>
第一条样本: {'prompt_id': 'L1_a', 'politeness_level': 1, 'paraphrase': 'a', 'politeness_classifier_score': 0.0, 'wrapper_prefix': "Score this passage. Don't waste my time with explanations.\n", 'wrapper_suffix': 'Just output the JSON object {"score": <0-3>} and nothing else.\n'}


In [20]:
import yaml
with open("config/prompts.yaml") as f:
    prompts = yaml.safe_load(f)

for item in prompts["variants"]:
    if item["prompt_id"] in ["L5_a", "L5_b", "L5_c"]:
        print("="*64)
        print(f"[{item['prompt_id']}]  politeness_level={item['politeness_level']}  "
              f"classifier_score={item['politeness_classifier_score']}")
        print("-"*64)
        print("PREFIX:", repr(item["wrapper_prefix"]))
        print("SUFFIX:", repr(item["wrapper_suffix"]))

[L5_a]  politeness_level=5  classifier_score=2.999
----------------------------------------------------------------
PREFIX: 'I would be incredibly grateful if you could kindly take a moment to evaluate the relevance of the passage below to the search query. Your expert judgment means a great deal to me.\n'
SUFFIX: 'If it isn\'t too much trouble, could you please respond with only the JSON object {"score": <0-3>}? Thank you so much in advance for your invaluable help.\n'
[L5_b]  politeness_level=5  classifier_score=2.999
----------------------------------------------------------------
PREFIX: 'If you would be so kind, I humbly request your thoughtful assessment of how relevant the passage below is to the given query. I truly appreciate the care you bring to this task.\n'
SUFFIX: 'Whenever convenient, would you please share your answer as only the JSON object {"score": <0-3>}? I am deeply thankful for your time and generous assistance.\n'
[L5_c]  politeness_level=5  classifier_score=2.99